## KNN и SVM без кросс-валидации

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.base import clone
import numpy as np

# Загрузка данных
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
X = train.drop(columns=["Id", "Survived"]).copy()
y = train["Survived"]
X_test = test.drop(columns="Id").copy()
for data in [X, X_test]:
    data["around_whealth"] = (data["Parch"] + data["SibSp"]) * data["Fare"]


## Подготовка признаков

In [2]:
# Подготовка признаков
numeric_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare", "around_whealth"]
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Пропуск")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_columns),
    ("categorical", categorical_transformer, ["Sex", "Embarked"]),
])


In [3]:
# Случайные параметры
seed = int(np.random.SeedSequence().entropy)
rng = np.random.default_rng(seed)
knn_params = {
    "n_neighbors": int(rng.integers(3, 41)),
    "weights": str(rng.choice(["uniform", "distance"])),
    "p": int(rng.choice([1, 2])),
}
svm_params = {
    "C": float(10 ** rng.uniform(-1, 2)),
    "gamma": float(10 ** rng.uniform(-3, -0.5)),
    "kernel": "rbf",
}
print("Зерно генератора:", seed)
print("KNN:", knn_params)
print("SVM:", svm_params)

# Обучение без кросс-валидации
models = {
    "knn": KNeighborsClassifier(**knn_params),
    "svm": SVC(**svm_params, random_state=42),
}
trained_models = {}
sample = pd.read_csv("data/sample_submission.csv")
for name, estimator in models.items():
    model = Pipeline([("preprocessor", clone(preprocessor)), ("model", estimator)])
    model.fit(X, y)
    submission = pd.DataFrame({"Id": test["Id"], "Survived": model.predict(X_test)})
    assert submission.columns.tolist() == sample.columns.tolist()
    assert submission["Id"].equals(sample["Id"])
    assert submission["Survived"].isin([0, 1]).all()
    assert not submission.isna().any().any()
    filename = f"submission_{name}_scaled_no_cv.csv"
    submission.to_csv(filename, index=False)
    trained_models[name] = model
    print("Сохранён:", filename)


Зерно генератора: 115867436361336123128560509005920121341
KNN: {'n_neighbors': 28, 'weights': 'uniform', 'p': 2}
SVM: {'C': 0.9256511739721229, 'gamma': 0.04172327592599367, 'kernel': 'rbf'}


Сохранён: submission_knn_scaled_no_cv.csv
Сохранён: submission_svm_scaled_no_cv.csv
